# STA4020 Part 1 · 浅层梯度提升树收益模型

**两种模型对全部 11 个候选因子统一筛选。** 锁池当天共同使用 6 项，两种模型各从 2,178 支股票中独立锁定 50 支，名单重合 13 支。截断到 2014-12-30 重算，新的树池名单、排名和初始预测一致。

**样本外 116 个月。** 在岭回归固定池上，树的 RMSE 为 15.672%，岭回归为 15.699%，Rank IC 分别为 0.1247 和 0.1288。各自在自己的固定池上按相同成交规则回测，树模型净年化 **12.36%**，岭回归 **12.08%**。

## 1. 与原版平行的流程

| 时点或环节 | 两条流程共用的规则 |
|---|---|
| 构造特征 | 在当期合格的沪深股票截面上计算同一套 11 项候选特征。 |
| 筛选特征 | 每次预测用最近 72 个标签已实现的训练月。全部 11 项按 Rank IC、6 个月块 bootstrap 和 BY 校正筛选，保留 $q\le0.10$ 的全部因子。 |
| 锁池 | 2014-12-30 收盘后，两种模型分别预测 2015 年 1 月收益，各取预测最高的 50 支。样本外从 2014-12-31 收盘开始。 |
| 更新 | 2015-01 至 2024-08 的 116 个目标月，每月更新同一训练窗口、重新筛特征并各自拟合。两套 50 支名单保持固定。 |
| 比较 | 先在同一批股票上比较两种预测，再交叉比较两套股票池与两套预测的回测结果。 |

树模型使用预先指定的深度 3、60 轮提升和 0.08 学习率。每棵子树最多有 8 个叶子，每个叶子至少有 250 条训练记录。提升逐轮拟合剩余误差，学习特征之间的非线性关系。这一部分参考 Pedersen《Machine Learning in Asset Pricing》第 23–28 页。

In [ ]:
# Install the notebook dependencies from the project requirements.txt.
from pathlib import Path

import h5py
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import brentq
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

cwd = Path.cwd().resolve()
PROJECT = next(
    (root for root in (cwd, *cwd.parents)
     if (root / "part1_return_model" / "notebooks").is_dir()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("Start the notebook from group-project or a subdirectory.")
DATA = PROJECT / "data"
BASELINE_OUTPUT = PROJECT / "artifacts" / "part1"
OUTPUT = BASELINE_OUTPUT / "tree"

FIRST_OOS_TARGET = pd.Period("2015-01", freq="M")
TRAIN_MONTHS = 72
RISK_MONTHS = 36
POOL_SIZE = 50
MODEL_HOLDINGS = 25
TRADE_COST_RATE = 0.0020  # Per yuan of stock bought or sold (20 bps)
assert 31 <= POOL_SIZE and 1 <= MODEL_HOLDINGS <= POOL_SIZE
MIN_FACTOR_PAIRS = 100
MIN_IC_MONTHS = 48
RIDGE_PENALTY = 0.10
FDR_Q = 0.10
BLOCK_MONTHS = 6
BOOTSTRAPS = 1999
SEED = 4020

TREE = "Shallow boosting"
FM = "Original FM ridge"
TREE_POOL = "Tree-selected 50"
FM_POOL = "FM-selected 50"
TREE_DEPTH = 3
TREE_THREADS = 2

plt.rcParams.update({
    "figure.dpi": 135,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})

## 2. 原始数据和同一套 11 个因子

以原始月收益为标签，日度复权收盘价、日成交额和上月末市值用于构造日、周、月度特征。资格规则使用信号日有效价格、当天交易状态、过去 12 个完整月的收益和正的滞后市值。因子在当期可评分宽截面内转为居中百分位分数。缺失值按原版填入中心排名 0。

11 个候选特征为市值、近 21 日平均成交额、63 日收益波动、8 周动量、12–1 月动量、上月末到信号日的收益、过去 5 日收益、12 周波动、上月收益、PCA 第一主成分暴露和剔除前三个主成分后的残差波动。

In [ ]:
def month_index(frame):
    frame = frame.copy()
    frame.index = pd.DatetimeIndex(frame.index).to_period("M")
    frame = frame.sort_index()
    assert frame.index.is_unique and frame.columns.is_unique
    return frame


def month_end(period):
    return period.end_time.normalize()


raw_returns = month_index(pd.read_pickle(DATA / "monthly_returns.pkl"))
market_cap = month_index(pd.read_pickle(DATA / "mcap.pkl"))
first_formation = FIRST_OOS_TARGET - 1
first_feature_month = first_formation - (TRAIN_MONTHS + 1)
last_target = raw_returns.index.max()
formation_months = pd.period_range(first_feature_month, last_target - 1, freq="M")
oos_months = formation_months[formation_months >= first_formation]
assert len(oos_months) >= 60
assert last_target >= FIRST_OOS_TARGET + 59
assert raw_returns.index.equals(pd.period_range(raw_returns.index[0], last_target, freq="M"))

with h5py.File(DATA / "price.h5", "r") as price_file:
    store = price_file["price"]
    trading_days = pd.DatetimeIndex(store["axis1"][:])
    price_columns = pd.Index(store["axis0"].asstr()[:], name="Asset")
    assert price_columns.equals(pd.Index(store["block0_items"].asstr()[:]))
    assert trading_days.is_unique and trading_days.is_monotonic_increasing
    start = trading_days.searchsorted((first_feature_month - 5).start_time)
    finish = trading_days.searchsorted(last_target.end_time, side="right")
    prices = pd.DataFrame(
        store["block0_values"][start:finish], index=trading_days[start:finish],
        columns=price_columns,
    )

assets = prices.columns.intersection(raw_returns.columns).sort_values()
assets = assets[assets.str.fullmatch(r"\d{6}\.(SZ|SH)")]
prices = prices.reindex(columns=assets).where(lambda values: values.gt(0))
returns = raw_returns.reindex(columns=assets).replace([np.inf, -np.inf], np.nan)
market_cap = market_cap.reindex(columns=assets)
assert (returns.stack() >= -1).all()

exchange_month = pd.Series(prices.index, index=prices.index).groupby(prices.index.to_period("M"))
last_close = exchange_month.last()
signal_close = exchange_month.nth(-2)
signal_close.index = signal_close.index.to_period("M")
assert (signal_close.reindex(formation_months) < last_close.reindex(formation_months)).all()

# Cross-check close-to-close return construction on all overlapping quotes.
end_prices = prices.loc[last_close.to_numpy()].set_axis(last_close.index)
implied_monthly = end_prices.pct_change(fill_method=None)
price_difference = (implied_monthly - returns.reindex(implied_monthly.index)).abs().stack()
assert price_difference.size > 100_000
display(pd.Series({
    "Shanghai / Shenzhen symbols in raw data": len(assets),
    "Overlapping price/return observations": price_difference.size,
    "Share matching to 1 bp": price_difference.le(0.0001).mean(),
    "First OOS target month": str(FIRST_OOS_TARGET),
    "Last OOS target month": str(last_target),
    "OOS target months": len(oos_months),
}, name="Source and calendar checks").to_frame())

In [ ]:
def restrictions_on_days(filename, days):
    records = pd.read_pickle(DATA / filename).index
    dates = records.get_level_values("date")
    selected = records[dates.isin(days)]
    return {
        pd.Timestamp(date): pd.Index(rows.index.get_level_values("asset").unique())
        for date, rows in selected.to_frame(index=True).groupby(level="date", sort=False)
    }


def load_daily_amount(days, asset_columns):
    amount = pd.read_pickle(DATA / "daily_amount.pkl")["total_turnover"]
    date_level = amount.index.levels[1]
    keep = date_level.isin(days)[amount.index.codes[1]]
    wide = amount.loc[keep].unstack("order_book_id")
    wide.columns = (
        wide.columns.str.replace(".XSHE", ".SZ", regex=False)
        .str.replace(".XSHG", ".SH", regex=False)
    )
    assert wide.index.is_unique and wide.columns.is_unique
    return wide.reindex(index=days, columns=asset_columns).astype("float32")


audit_days = pd.Index(signal_close.reindex(formation_months)).union(
    pd.Index(last_close.reindex(oos_months))
)
blacklist = restrictions_on_days("BLACKLIST.pkl", audit_days)
untradable = restrictions_on_days("UNTRADABLE.pkl", audit_days)
daily_amount = load_daily_amount(prices.index, assets)
assert daily_amount.notna().sum().sum() > 100_000

daily_returns = prices.pct_change(fill_method=None)
week_end_prices = (
    prices.loc[pd.Series(prices.index).groupby(prices.index.to_period("W-FRI")).last().to_numpy()]
    .set_axis(pd.Index(prices.index.to_period("W-FRI").unique().end_time.normalize()))
)
weekly_returns = week_end_prices.pct_change(fill_method=None)

ff5_factors = month_index(pd.read_csv(DATA / "FF5.csv", index_col="date", parse_dates=True))
daily_rf = pd.read_csv(DATA / "risk_free.csv", index_col="date", parse_dates=True)["RF"]
assert daily_rf.index.is_unique
required_rf = daily_rf.reindex(trading_days)
required_rf = required_rf.loc[(first_feature_month - RISK_MONTHS).start_time:last_target.end_time]
assert required_rf.notna().all()
rf_monthly = (1 + required_rf).groupby(required_rf.index.to_period("M")).prod(min_count=1) - 1

In [ ]:
def pca_exposures(history):
    complete = history.dropna(axis=1)
    assert complete.shape[1] >= MIN_FACTOR_PAIRS
    centered = complete.to_numpy() - complete.to_numpy().mean(axis=0)
    _, directions = np.linalg.eigh(centered @ centered.T)
    top = directions[:, -3:][:, ::-1]
    market_direction = centered.mean(axis=1)
    direction = top[:, 0] * (1 if top[:, 0] @ market_direction >= 0 else -1)
    # Scale PC1 to the volatility of the equal-weight reference return.
    factor = direction * np.linalg.norm(market_direction)
    loading = (factor @ centered) / (factor @ factor)
    residual = centered - top @ (top.T @ centered)
    pca_features = pd.DataFrame({
        "pc1_market_beta": loading,
        "pca_residual_volatility": residual.std(axis=0, ddof=1),
    }, index=complete.columns)
    return pca_features


def formation_inputs(month):
    as_of = signal_close.loc[month]
    monthly_history = returns.loc[month - RISK_MONTHS:month - 1]
    recent_amount = daily_amount.loc[:as_of].tail(21)
    previous_cap = market_cap.loc[month - 1]
    blocked = blacklist.get(as_of, pd.Index([])).union(untradable.get(as_of, pd.Index([])))
    eligible = (
        prices.loc[as_of].notna()
        & previous_cap.gt(0)
        & monthly_history.tail(12).notna().all(axis=0)
        & ~assets.isin(blocked)
    )
    eligible_assets = assets[eligible]
    assert len(eligible_assets) >= max(POOL_SIZE, MIN_FACTOR_PAIRS)
    monthly_history = monthly_history.loc[:, eligible_assets]
    pca_features = pca_exposures(monthly_history)

    daily = daily_returns.loc[:as_of].tail(63).loc[:, eligible_assets]
    weekly = weekly_returns.loc[:as_of].tail(12).loc[:, eligible_assets]
    mean_amount = recent_amount.loc[:, eligible_assets].mean().where(
        recent_amount.loc[:, eligible_assets].count().ge(15)
    )
    prior_close = last_close.loc[month - 1]
    raw = pd.DataFrame({
        "log_size": np.log(previous_cap.loc[eligible_assets]),
        "log_amount_21d": np.log1p(mean_amount.where(mean_amount.ge(0))),
        "return_5d": (1 + daily.tail(5)).prod(min_count=5) - 1,
        "volatility_63d": daily.std().where(daily.count().ge(50)),
        "momentum_8w": (1 + weekly.tail(8)).prod(min_count=8) - 1,
        "volatility_12w": weekly.std().where(weekly.count().ge(10)),
        "momentum_12_1m": (1 + monthly_history.iloc[-12:-1]).prod(min_count=11) - 1,
        "lagged_month_return": monthly_history.iloc[-1],
        "return_month_to_signal": (
            prices.loc[as_of, eligible_assets] / prices.loc[prior_close, eligible_assets] - 1
        ),
    }).join(pca_features).replace([np.inf, -np.inf], np.nan)

    # Mid-rank transformation. First rank maps just above the lowest endpoint.
    ranked = raw.rank(method="average").sub(0.5).div(raw.count())
    features = (np.sqrt(12) * (ranked - 0.5)).astype("float32")
    assert features.shape[1] == 11
    return features

## 3. 每月用同一个 72 个月窗口筛选并拟合

每个历史截面用当月因子分数配对次月实际收益。Rank IC 使用至少 100 对股票，每个因子至少有 48 个月的有效 IC，才能做显著性检验。每月用 72 个历史截面重算全部 11 项的 q 值并采用同一门槛，入选数随窗口变化。两个模型在同一预测月读到相同的训练月和相同的入选因子。

树模型把 72 个截面放在一起训练。每月先对训练收益做 1%/99% 缩尾，再减去该月截面均值，让树学习同月股票之间的收益差。给每个训练月相同的总权重。在当期宽截面内将树输出居中，加回最近 72 个已实现截面的历史均值，得到次月总收益预测。Fama–MacBeth 基准重现原版的逐月岭回归和系数平均。

某月若没有因子通过，导出共同收益预测，当月的模型组合以现金为目标，成交仍遵守冻结持仓规则。

In [ ]:
def block_draws(n_dates, seed=SEED):
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n_dates / BLOCK_MONTHS))
    starts = rng.integers(n_dates, size=(BOOTSTRAPS, n_blocks, 1))
    indices = (starts + np.arange(BLOCK_MONTHS)) % n_dates
    return indices.reshape(BOOTSTRAPS, -1)[:, :n_dates]


def by_adjust(p_values):
    valid = p_values.dropna().sort_values()
    m = len(valid)
    ranks = np.arange(1, m + 1)
    scaled = valid.to_numpy() * m * np.sum(1 / ranks) / ranks
    adjusted = np.minimum.accumulate(scaled[::-1])[::-1].clip(0, 1)
    return pd.Series(adjusted, index=valid.index).reindex(p_values.index)


def screen_factors(ic_history):
    means = ic_history.mean()
    centered = ic_history.sub(means).to_numpy()
    with np.errstate(invalid="ignore", divide="ignore"):
        samples = centered[block_draws(len(ic_history))]
        sampled_means = np.nansum(samples, axis=1) / np.isfinite(samples).sum(axis=1)
    valid_draws = np.isfinite(sampled_means).sum(axis=0)
    extreme = (np.abs(sampled_means) >= means.abs().to_numpy()).sum(axis=0)
    p_values = pd.Series((extreme + 1) / (valid_draws + 1), index=means.index)
    p_values = p_values.where(ic_history.count().ge(MIN_IC_MONTHS))
    summary = pd.DataFrame({
        "mean_rank_ic": means,
        "months": ic_history.count(),
        "p_value": p_values,
        "q_value": by_adjust(p_values),
    })
    selected = summary.loc[summary.q_value.le(FDR_Q)].sort_values("q_value", kind="stable")
    return selected.index.to_list(), summary

In [ ]:
def rank_ic(signal, target, minimum=MIN_FACTOR_PAIRS):
    measured = signal.notna() & target.notna()
    signal, target = signal.loc[measured], target.loc[measured]
    if len(signal) < minimum or signal.nunique() < 2 or target.nunique() < 2:
        return np.nan
    return float(spearmanr(signal, target).statistic)


def training_statistics(features, target):
    valid = target.notna()
    target = target.loc[valid]
    assert len(target) >= MIN_FACTOR_PAIRS
    factor_ic = features.loc[valid].apply(rank_ic, target=target)
    design = features.loc[valid].fillna(0).to_numpy(dtype=float)
    low, high = target.quantile([0.01, 0.99])
    response = target.clip(low, high).to_numpy()
    design_mean, response_mean = design.mean(axis=0), response.mean()
    centered_design, centered_response = design - design_mean, response - response_mean
    # Cache moments so different screened subsets can reuse the same monthly sample.
    return {
        "ic": factor_ic,
        "design_mean": design_mean,
        "response_mean": response_mean,
        "gram": centered_design.T @ centered_design / len(target),
        "cross": centered_design.T @ centered_response / len(target),
        "raw_return_mean": target.mean(),
        "tree_design": design.astype("float32"),
        "tree_response": centered_response.astype("float32"),
    }


def fama_macbeth_ridge(train_stats, factor_names, feature_columns):
    if not factor_names:
        return np.array([np.mean([stats["response_mean"] for stats in train_stats])])
    selected = feature_columns.get_indexer(factor_names)
    assert (selected >= 0).all()
    penalty = np.eye(len(selected)) * RIDGE_PENALTY
    monthly_coefficients = []
    for stats in train_stats:
        slopes = np.linalg.solve(
            stats["gram"][np.ix_(selected, selected)] + penalty, stats["cross"][selected]
        )
        intercept = stats["response_mean"] - stats["design_mean"][selected] @ slopes
        monthly_coefficients.append(np.r_[intercept, slopes])
    return np.mean(monthly_coefficients, axis=0)

In [ ]:
feature_cache = {}
training_cache = {}


def features_at(formation):
    if formation not in feature_cache:
        feature_cache[formation] = formation_inputs(formation)
    return feature_cache[formation]


def training_months_at(formation):
    train = pd.period_range(formation - TRAIN_MONTHS - 1, formation - 2, freq="M")
    assert train.isin(formation_months).all()
    assert last_close.loc[train + 1].lt(signal_close.loc[formation]).all()
    return train


def training_window(formation):
    train = training_months_at(formation)
    for past in train:
        if past not in training_cache:
            features = features_at(past)
            realized = returns.loc[past + 1].reindex(features.index)
            training_cache[past] = training_statistics(features, realized)
    return [training_cache[past] for past in train]


def predict_tree(history, features, columns):
    common_level = float(np.mean([stats["response_mean"] for stats in history]))
    if not columns:
        return np.full(len(features), common_level), common_level
    selected = features.columns.get_indexer(columns)
    assert (selected >= 0).all()
    blocks = [stats["tree_design"][:, selected] for stats in history]
    design = np.concatenate(blocks)
    target = np.concatenate([stats["tree_response"] for stats in history])
    observations_per_month = np.array([len(block) for block in blocks])
    per_row_weight = np.repeat(
        len(target) / (len(blocks) * observations_per_month),
        observations_per_month,
    )
    model = HistGradientBoostingRegressor(
        max_depth=TREE_DEPTH, max_leaf_nodes=2**TREE_DEPTH, max_iter=60,
        learning_rate=0.08, min_samples_leaf=250, l2_regularization=5.0, max_bins=32,
        early_stopping=False, random_state=SEED,
    )
    with threadpool_limits(limits=TREE_THREADS):
        model.fit(design, target, sample_weight=per_row_weight)
        premium = model.predict(features.loc[:, columns].fillna(0).to_numpy())
    return common_level + (premium - premium.mean()), common_level


def forecast_month(formation):
    history = training_window(formation)
    ic_history = pd.DataFrame([stats["ic"] for stats in history])
    selected_factors, factor_audit = screen_factors(ic_history)
    features = features_at(formation)
    tree_prediction, tree_level = predict_tree(history, features, selected_factors)
    fm_coef = fama_macbeth_ridge(history, selected_factors, features.columns)
    fm_prediction = fm_coef[0] + features[selected_factors].fillna(0) @ fm_coef[1:]
    forecasts = pd.DataFrame({TREE: tree_prediction, FM: fm_prediction}, index=features.index)
    assert np.isfinite(forecasts.to_numpy()).all()
    fallback = {TREE: tree_level, FM: np.mean([stats["raw_return_mean"] for stats in history])}
    return forecasts, fallback, selected_factors, factor_audit


def descending_scores(score):
    return score.sort_index().sort_values(ascending=False, kind="stable")

## 4. 在样本外前独立锁定两个池

树分数取前 50 生成树池。Fama–MacBeth 分数独立取前 50 复现原池。首次训练与筛选完成时，最新收益标签为 2014-11-28 收盘后的 11 月收益。锁池计算只访问当时已成熟的训练标签。入池预测分数并列时按股票代码从小到大排。

In [ ]:
oos_start = last_close.loc[first_formation]
lock_signal_date = signal_close.loc[first_formation]
initial_forecasts, initial_levels, initial_factors, initial_factor_audit = forecast_month(first_formation)
if not initial_factors:
    raise ValueError("No factor passed the pre-OOS screen; the tree pool cannot be ranked.")
latest_lock_label = last_close.loc[training_months_at(first_formation)[-1] + 1]
assert latest_lock_label < lock_signal_date < oos_start
assert len(training_cache) == TRAIN_MONTHS
assert all(last_close.loc[p + 1] < lock_signal_date for p in training_cache)

tree_universe = descending_scores(initial_forecasts[TREE]).head(POOL_SIZE).index
fm_universe = descending_scores(initial_forecasts[FM]).head(POOL_SIZE).index
universes = {TREE_POOL: tree_universe, FM_POOL: fm_universe}
assert all(len(pool) == POOL_SIZE and pool.is_unique for pool in universes.values())

tree_pool = pd.DataFrame({
    "Asset": tree_universe,
    "Selection_Rank": np.arange(1, POOL_SIZE + 1),
    "Selection_Signal_Date": lock_signal_date,
    "OOS_Start_Close": oos_start,
    "Last_Selection_Training_Label_Date": latest_lock_label,
    "Initial_Target_Date": month_end(FIRST_OOS_TARGET),
    "Initial_Expected_Return": initial_forecasts.loc[tree_universe, TREE].to_numpy(),
})

overlap = len(tree_universe.intersection(fm_universe))
display(pd.Series({
    "Selection date": lock_signal_date, "Last training outcome": latest_lock_label,
    "OOS entry close": oos_start, "Stocks scored": len(initial_forecasts),
    "Locked names per model": POOL_SIZE,
    "Tree / FM common names": overlap,
}, name="Pre-OOS lock audit").to_frame())
display(initial_factor_audit.assign(used_at_lock=lambda f: f.index.isin(initial_factors))
        .sort_values("q_value").style.format({"mean_rank_ic": "{:+.4f}", "q_value": "{:.4f}",
                                              "p_value": "{:.4f}"}))
display(tree_pool[["Asset", "Selection_Rank", "Initial_Expected_Return"]].head())

## 5. 固定池上的样本外预测比较

在树池和原池各做一次同股票、同月对比。评分使用有当期信号且当月实现收益可用的共同样本。每月先等权计算股票误差，再等权汇总各月。RMSE 衡量收益数值误差，Rank IC 衡量同月排序。覆盖与因子数另外记录。

In [ ]:
forecast_history = {}
tree_export = {}
fm_reproduced = {}
factor_history = {}
for month in oos_months:
    if month == first_formation:
        forecasts, levels, factors = initial_forecasts, initial_levels, initial_factors
    else:
        forecasts, levels, factors, _ = forecast_month(month)
    forecast_history[month] = forecasts
    factor_history[month] = factors
    tree_export[month_end(month)] = forecasts[TREE].reindex(tree_universe).fillna(levels[TREE])
    fm_reproduced[month_end(month)] = forecasts[FM].reindex(fm_universe).fillna(levels[FM])

# Optional parity check against the updated main FM notebook exports.
main_pool_file = BASELINE_OUTPUT / "P1_Candidate_Universe.csv"
main_forecast_file = BASELINE_OUTPUT / "P1_Expected_Returns.csv"
if main_pool_file.is_file() and main_forecast_file.is_file():
    prior_pool = pd.read_csv(main_pool_file).sort_values("Selection_Rank")
    reference = pd.read_csv(main_forecast_file, index_col="Date", parse_dates=True)
    reproduced = pd.DataFrame(fm_reproduced).T
    assert prior_pool.Asset.tolist() == fm_universe.tolist(), (
        "FM pool mismatch: rerun Part1_Return_Model.ipynb before the tree notebook."
    )
    assert reference.columns.tolist() == reproduced.columns.tolist()
    assert reference.index.equals(reproduced.index)
    np.testing.assert_allclose(reference.to_numpy(), reproduced.to_numpy(), rtol=1e-10, atol=1e-12)
    print("FM pool and monthly forecasts match the main notebook exports.")

print("Factors at lock:", ", ".join(initial_factors))
factor_count = pd.Series({month_end(month): len(factors) for month, factors in factor_history.items()})
frequency = pd.Series([factor for factors in factor_history.values() for factor in factors]).value_counts()
display(pd.DataFrame({
    "Used at lock": initial_factor_audit.index.isin(initial_factors),
    "Selected OOS months": frequency.reindex(initial_factor_audit.index, fill_value=0),
}, index=initial_factor_audit.index).sort_values("Selected OOS months", ascending=False))
display(factor_count.value_counts().sort_index().rename("OOS months").rename_axis("Selected factors").to_frame())

In [ ]:
prediction_rows = []
for month, forecasts in forecast_history.items():
    for pool_name, members in universes.items():
        eligible = members[members.isin(forecasts.index)]
        realization = returns.loc[month + 1, eligible].dropna()
        common = forecasts.loc[realization.index]
        assert len(realization) >= 2
        for name, prediction in common.items():
            prediction_rows.append({
                "Pool": pool_name, "Target_Date": month_end(month + 1), "Model": name,
                "MSE": (prediction - realization).pow(2).mean(),
                "Rank IC": rank_ic(prediction, realization, minimum=2),
                "Common stocks": len(realization),
            })

monthly_accuracy = pd.DataFrame(prediction_rows)
accuracy = monthly_accuracy.groupby(["Pool", "Model"], sort=False)[
    ["MSE", "Rank IC", "Common stocks"]
].mean()
accuracy["Monthly RMSE"] = np.sqrt(accuracy.pop("MSE"))
display(accuracy[["Monthly RMSE", "Rank IC", "Common stocks"]].style.format({
    "Monthly RMSE": "{:.3%}", "Rank IC": "{:+.4f}", "Common stocks": "{:.1f}",
}))


def block_interval(values):
    samples = values.to_numpy()[block_draws(len(values), seed=SEED + 1)]
    return np.quantile(samples.mean(axis=1), [0.025, 0.975])


mse_differences = {}
for name in universes:
    errors = monthly_accuracy.loc[monthly_accuracy.Pool.eq(name)].pivot(
        index="Target_Date", columns="Model", values="MSE"
    )
    advantage = (errors[FM] - errors[TREE]) / errors[FM].mean()
    mse_differences[name] = [advantage.mean(), *block_interval(advantage)]
mse_intervals = pd.DataFrame.from_dict(
    mse_differences, orient="index", columns=["MSE reduction vs FM", "95% low", "95% high"]
)
display(mse_intervals.style.format("{:+.2%}"))

## 6. 用同样的交易规则比较组合

信号日每个模型在指定的固定池内选前 25 支，每支目标权重 4%。等权对照给池内有效信号股每支目标权重 2%。按主 notebook 的规则处理现金、不可交易的冻结仓位、黑名单和收盘重平衡，买卖每笔成交金额各收 20 bp。市场参照采用中证 500 价格指数和 FF5 的市场收益加无风险收益。

同时模拟两个交叉组合。在固定股票池下交换预测模型，有助于区分排序和初始选池的贡献。实际收益缺失时，持仓按最新账面值记账，回测单独统计受影响的仓位。预测精度表只用实际收益可用的样本。

In [ ]:
CSI500 = "CSI 500 price index"
MARKET_PROXY = "FF5 market + RF"

index_close = pd.read_pickle(DATA / "zz500.pk")["close"].sort_index()
assert isinstance(index_close.index, pd.DatetimeIndex)
assert index_close.index.is_unique
comparison_months = pd.period_range(first_formation, last_target, freq="M")
index_monthly = index_close.reindex(last_close.loc[comparison_months].to_numpy())
assert index_monthly.notna().all() and index_monthly.gt(0).all()
index_monthly.index = comparison_months
csi500_return = index_monthly.pct_change(fill_method=None)
broad_market_return = ff5_factors["MKT"] + rf_monthly


def execute_targets(pretrade, desired, frozen):
    # All values are fractions of pretrade NAV. Protect frozen holdings and
    # solve for fees so the post-trade stock plus cash allocation is self-financing.
    frozen_value = pretrade[frozen].sum()
    free_target = desired.mask(frozen, 0.0)
    target_total = free_target.sum()

    def stock_positions(fee):
        available = max(0.0, 1 - frozen_value - fee)
        scale = min(1.0, available / target_total) if target_total else 0.0
        return pretrade.where(frozen, free_target * scale)

    def traded_fraction(positions):
        return (positions - pretrade).abs().sum()

    fee = brentq(
        lambda estimate: estimate - TRADE_COST_RATE * traded_fraction(stock_positions(estimate)),
        0.0, 2 * TRADE_COST_RATE + 1e-10,
    )
    positions = stock_positions(fee)
    risky_weights = positions / (1 - fee)
    cash_weight = max(0.0, 1 - risky_weights.sum())
    assert risky_weights.ge(0).all()
    assert np.isclose(risky_weights.sum() + cash_weight, 1.0)
    assert np.allclose(positions[frozen], pretrade[frozen])
    return risky_weights, cash_weight, fee, traded_fraction(positions)




portfolio_specs = {
    "Tree on tree pool": (TREE_POOL, TREE),
    "FM on FM pool": (FM_POOL, FM),
    "Tree on FM pool": (FM_POOL, TREE),
    "FM on tree pool": (TREE_POOL, FM),
    "Equal weight on tree pool": (TREE_POOL, None),
    "Equal weight on FM pool": (FM_POOL, None),
}

pretrade = {
    name: pd.Series(0.0, index=universes[pool]) for name, (pool, _) in portfolio_specs.items()
}
portfolio_rows = []
execution_rows = []
for formation, forecasts in forecast_history.items():
    target_month = formation + 1
    entry_day = last_close.loc[formation]
    cash_return = rf_monthly.loc[target_month]
    date = month_end(target_month)
    row = {"Target_Date": date, CSI500: csi500_return.loc[target_month],
           MARKET_PROXY: broad_market_return.loc[target_month]}
    for name, (pool_name, scorer) in portfolio_specs.items():
        members = universes[pool_name]
        eligible = members[members.isin(forecasts.index)]
        desired = pd.Series(0.0, index=members)
        if scorer is None:
            desired.loc[eligible] = 1 / POOL_SIZE
        elif factor_history[formation]:
            orders = descending_scores(forecasts.loc[eligible, scorer]).head(MODEL_HOLDINGS).index
            desired.loc[orders] = 1 / MODEL_HOLDINGS

        frozen = prices.loc[entry_day, members].isna()
        frozen |= members.isin(untradable.get(entry_day, pd.Index([])))
        desired.loc[members.isin(blacklist.get(entry_day, pd.Index([])))] = 0.0
        weights, cash, fee, turnover = execute_targets(pretrade[name], desired, frozen)
        realized = returns.loc[target_month, members]
        unavailable_weight = weights.loc[realized.isna()].sum()
        marked_returns = realized.fillna(0.0)
        gross = float(weights @ marked_returns + cash * cash_return)
        assert gross > -1
        row[name] = (1 - fee) * (1 + gross) - 1
        pretrade[name] = weights.mul(1 + marked_returns) / (1 + gross)
        execution_rows.append({
            "Target_Date": date, "Portfolio": name, "Fee": fee,
            "Two-way turnover": turnover, "Cash": cash, "Frozen weight": weights.loc[frozen].sum(),
            "Weight with missing actual return": unavailable_weight,
        })
    portfolio_rows.append(row)

portfolio_returns = pd.DataFrame(portfolio_rows).set_index("Target_Date")
assert portfolio_returns.notna().all().all() and len(portfolio_returns) >= 60
execution = pd.DataFrame(execution_rows)
display(execution.groupby("Portfolio", sort=False)[
    ["Two-way turnover", "Cash", "Frozen weight", "Weight with missing actual return"]
].mean().style.format("{:.2%}"))

In [ ]:
wealth = (1 + portfolio_returns).cumprod()
drawdown = wealth / wealth.cummax().clip(lower=1) - 1
performance = pd.DataFrame({
    "Annualized return": wealth.iloc[-1].pow(12 / len(wealth)) - 1,
    "Maximum drawdown": drawdown.min(),
    "Cumulative return": wealth.iloc[-1] - 1,
})
display(performance.style.format("{:.2%}"))

main_comparison = portfolio_returns["Tree on tree pool"] - portfolio_returns["FM on FM pool"]
lo, hi = block_interval(main_comparison)
display(pd.Series({
    "Tree vs FM own pools, monthly spread": main_comparison.mean(),
    "Paired 95% low": lo, "Paired 95% high": hi,
}, name="Net spread").to_frame().style.format("{:+.2%}"))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), layout="constrained")
initial_value = pd.DataFrame(1.0, index=[oos_start], columns=wealth.columns)
curves = pd.concat([initial_value, wealth])
for name, color, style in [
    ("Tree on tree pool", "#007C72", "-"), ("FM on FM pool", "#B35F26", "--"),
    (CSI500, "#696378", "-."), (MARKET_PROXY, "#4D6889", ":"),
]:
    axes[0].plot(curves[name], color=color, linestyle=style, label=name, lw=1.8)
axes[0].set(title="Net performance with independently frozen stock pools",
            ylabel="Value of 1 yuan at OOS start")
axes[0].legend(frameon=False, ncols=2, loc="upper left")

same_pool = monthly_accuracy.loc[monthly_accuracy.Pool.eq(FM_POOL)].pivot(
    index="Target_Date", columns="Model", values="Rank IC"
)
for name, color, style in [(TREE, "#007C72", "-"), (FM, "#B35F26", "--")]:
    axes[1].plot(same_pool[name].rolling(12).mean(), color=color, linestyle=style,
                 label=name, lw=1.8)
axes[1].axhline(0, color="#777777", lw=0.8)
axes[1].set(title="Ranking accuracy on the same 50 stocks selected by the original model",
            xlabel="Target month", ylabel="12-month mean Rank IC")
axes[1].legend(frameon=False, loc="lower right")
for axis in axes:
    axis.xaxis.set_major_locator(mdates.YearLocator(2))
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.show()

### 结果怎么读

- **同池预测。** 在岭回归池，树的相对 MSE 改善为 0.34%，配对 95% 区间为 −0.10% 至 +0.88%。在树池，MSE 改善为 0.25%，配对区间也跨过零。岭回归在两个池中的平均 Rank IC 都略高。
- **两套独立池的回测。** 树模型净年化 12.36%，比中证 500 价格指数和 FF5 市场代理分别高 13.77 和 10.33 个百分点。岭回归净年化 12.08%，相同基准下分别高 13.49 和 10.06 个百分点。树与岭回归在各自池上的月均净收益差为 −0.03 个百分点，95% 配对区间为 −0.51 至 +0.41 个百分点。
- **交叉回测。** 树在岭回归池上的净年化为 15.34%，岭回归在树池上为 11.71%。这两组用于观察池内排序和股票池组成的共同影响。

## 7. 树模型的三份输出

输出到 part1_output/tree/，包括固定 50 支名单、样本外次月预测宽表和同池实际收益宽表。预测表的 Date 是调仓月末，实际表的 Date 是实现月末。匹配实现收益时把预测日期加一个月。缺少当期有效信号的股票留在 50 列内，预测用树模型的共同收益水平回退，在策略里按交易规则处理。

In [ ]:
expected_wide = pd.DataFrame(tree_export).T.reindex(columns=tree_universe)
expected_wide.index.name = "Date"
actual_wide = returns.reindex(index=oos_months + 1, columns=tree_universe)
actual_wide.index = pd.DatetimeIndex([month_end(month) for month in actual_wide.index], name="Date")
assert expected_wide.shape == actual_wide.shape == (len(oos_months), POOL_SIZE)
assert expected_wide.columns.equals(actual_wide.columns)
assert tree_pool.Asset.to_list() == expected_wide.columns.to_list()
assert (expected_wide.index.to_period("M") + 1).equals(actual_wide.index.to_period("M"))
np.testing.assert_allclose(expected_wide.iloc[0], tree_pool.Initial_Expected_Return)
assert np.isfinite(expected_wide.to_numpy()).all()

OUTPUT.mkdir(parents=True, exist_ok=True)
tree_pool.to_csv(OUTPUT / "P1_Candidate_Universe.csv", index=False)
expected_wide.to_csv(OUTPUT / "P1_Expected_Returns.csv")
actual_wide.to_csv(OUTPUT / "Actual_Returns.csv")
print(f"Exported tree-selected universe and {len(expected_wide)} months × {POOL_SIZE} stocks to {OUTPUT}")
display(expected_wide.iloc[:3, :6].style.format("{:+.2%}"))